## 1. 聚宽

### 1.1 RSRS因子

📌 RSRS Factor Strategy Summary
- Goal: Identify trend strength and reliability using high–low price regression.
- Core Idea:Use linear regression between a stock/index’s high prices and low prices over a rolling window to capture upward momentum and price stability.

Step-by-Step Strategy:
- 📥 Data Input: Collect n + m days of high/low price data (e.g., n=18, m=1100).
- 📈 Linear Regression: Apply OLS (Ordinary Least Squares) regression: high ~ low.Do this over a rolling window of n days.
- 🧮 Extract Parameters:
    - β (slope): Indicates how steep the uptrend is.
    - R² (R-squared): Indicates how reliable the trend is (closer to 1 = better fit).

🔁 Z-score Standardization:
- Standardize the most recent β using the last m values: Z = \frac{\beta_t - \mu}{\sigma}

📊 Composite RSRS Score:
- Calculate final score using: RSRS\_score = Z \times \beta \times R^2
- Combines trend strength (β) and stability (R²).

📉 Trading Logic:
- If RSRS_score > 0.7 and available cash > 0 → Buy.
- If RSRS_score < -0.7 and holding position → Sell all.
- Execution:
    - Runs daily at market open.
    - Applied on a single security (e.g., 000300.XSHG, CSI 300 Index).
    - Can be extended to multiple stocks with looped RSRS rankings.

In [ ]:
# 导入函数库
from jqdata import *
import pandas as pd
import numpy as np
import statsmodels.api as sm

def initialize(context):
    set_benchmark('000300.XSHG')
    set_option('use_real_price', True)
    set_order_cost(OrderCost(close_tax=0.001, open_commission=0.0003, close_commission=0.0003, min_commission=5), type='stock')
    g.security = '000300.XSHG'
    g.buy_beta, g.sell_beta = 0.7, -0.7
    g.is_first, g.beta_list, g.r2_list = True, [], []
    run_daily(market_open, time='open', reference_security='000300.XSHG')

def calculate_rsrs(end_date='', n=18, m=1100):
    # n: 一阶拟合样本数，m:求均值方差样本数; 采用普通最小二乘法拟合，前面算过的就不重复计算了
    if g.is_first:
        df = get_price(g.security, end_date=end_date, count=m+n, frequency='daily', fields=['high','low'])
    else:
        df = get_price(g.security, end_date=end_date, count=n, frequency='daily', fields=['high','low'])
    for i in range(len(df))[(-m if g.is_first else -1):]:
        x = sm.add_constant(df['low'][i-n+1:i+1])
        y = df['high'][i-n+1:i+1]
        model = sm.OLS(y, x).fit()
        beta = model.params[1]
        r2 = model.rsquared
        g.beta_list.append(beta)
        g.r2_list.append(r2)  
    section = g.beta_list[-m:]
    mu = np.mean(section)
    sigma = np.std(section)
    z_score = (section[-1] - mu)/sigma
    z_score_right = z_score * beta * r2
    g.is_first = False
    return z_score_right

def market_open(context):
    cash = context.portfolio.available_cash
    z_score_right = calculate_rsrs(context.previous_date)
    if z_score_right>g.buy_beta and cash>0:
        order_value(g.security, cash)
    elif z_score_right<g.sell_beta and context.portfolio.positions[g.security].closeable_amount > 0:
        order_target(g.security, 0)

## 2. QMT

### 2.1 MACD因子

1．EMA（Exponential Moving Average）

EMA $t=\alpha \cdot P_t+(1-\alpha) \cdot \operatorname{EMA} t-1$

其中：
－$P_t$ ：第 $t$ 天的收盘价
．$\alpha=\frac{2}{N+1}:$ 平滑系数（ $N$ 为周期，如12或26）

2．DIF（快线）

DIF $t=$ EMAfast,$t-\mathrm{EMA}_{\text {slow }, t}$
－通常：
－fast $=12$
－slow $=26$

3．DEA（慢线）
$\mathrm{DEA}_t=\mathrm{EMA}_9\left(\mathrm{DIF}_t\right)$

即对 DIF 做一次 9 日指数移动平均。

4．MACD 柱状图
$$
\mathrm{MACD}_t=2 \cdot\left(\mathrm{DIF}_t-\mathrm{DEA}_t\right)
$$


注意：有些软件如 TradingView 不乘 2，因此 MACD 值可能不同，但走势一致。

递推逻辑总览

初始条件（通常）：
． $\mathrm{EMA}_{\text {fast }, 1}=P_1$
． $\mathrm{EMA}_{\text {slow }, 1}=P_1$
－ $\mathrm{DEA}_1=0$

之后逐日计算。

金叉与死叉判断条件
－金叉（买入信号）：

$$
\mathrm{DIF} t-1<\mathrm{DEA} t-1 \quad \text { and } \quad \mathrm{DIF}_t \geq \mathrm{DEA}_t
$$

－死叉（卖出信号）：

$$
\mathrm{DIF} t-1>\mathrm{DEA} t-1 \quad \text { and } \quad \mathrm{DIF}_t \leq \mathrm{DEA}_t
$$


In [ ]:
#encoding:gbk
import talib
import datetime
import numpy as np

# 《macd指标》
# qmt量化交易
# 金叉买入，死叉卖出
# 短时间内不重复下单
# 作者威♥: yiyou888wx
# (可开户、代写)
# 量化优势：对抗人性、不用盯盘

class a():
	pass

A = a()

def init(C):
	# 设置股票代码，上海SH结尾，深圳SZ结尾
	A.stock = '000333.SZ'
	# 填写自己的账号
	A.account = '39001785'
	# 设置每次交易的股票数量
	A.volume = 100
	# 设置k线周期
	A.period = '1h'
	# 1小时(3600秒)内不重复下单
	A.trade_time = 3600
	# macd参数配置
	# 分别是短周期、长周期、移动平均周期
	A.param1,A.param2,A.param3 = 12, 26, 9

    # 代码自动下载有用的历史数据，这样无需手动下载
	end = datetime.date.today()
	# 根据你的周期来，days可需改
	start = end - datetime.timedelta(days=10)
	start = start.strftime('%Y%m%d')
	end = end.strftime('%Y%m%d')
	download_history_data(A.stock, A.period, start, end)
	
	# 账号、资金情况
	account = get_trade_detail_data(A.account, 'STOCK', 'account')
	if len(account)==0:
		print(f'账号{A.account} 未登录 请检查')
		return
	account = account[0]
	# 可用资金
	available_cash = int(account.m_dAvailable)
	# 总资金
	total_cash = int(account.m_dBalance)
	print('【【账户信息】】')
	print(f'账户:{A.account},  总资金:{total_cash}, 可用资金:{available_cash}')

	# 初始化上次成交时间为 None
	A.last_trade_time = None


def handlebar(C):
	# 只处理最新数据传来的时候
	if not C.is_last_bar():
		return

	now = datetime.datetime.now()
	now_time = now.strftime('%H%M%S')
	#跳过非交易时间
	if now_time < '093000' or now_time > "150000":
		return

	stock_data_df = C.get_market_data(['close'],[A.stock], A.period, count=50)
	stock_data_closes = np.array(stock_data_df['close'])
	#print(f'stock_data_closes:{stock_data_closes[-5:]}')
	print(f'股票：{A.stock}，当前价格：{stock_data_closes[-1]:.2f}')
	
	# 数据太少 报错！
	if len(stock_data_closes)<50: 
		print('【错误】k线数据不足，无法计算！')
		return
	dif, dea, macd = talib.MACD(stock_data_closes, fastperiod=A.param1, 
								slowperiod=A.param2, signalperiod=A.param3)
	#print(f'macd: {macd[-5:]}')
	
	# 当前时间
	current_time = datetime.datetime.now()

	# 金叉
	if macd[-2]<0 and macd[-1]>=0:
		print(f'进入金叉状态')
		# 如果 last_trade_time 为 None，或者距离上次成交时间超过1小时
		if A.last_trade_time is None or (current_time - A.last_trade_time).total_seconds() > A.trade_time:
			print(f'买入{A.stock}股票，数量：{A.volume}，请保证资金充足!')
			passorder(23,1101,A.account,A.stock,14,-1,A.volume,2,C)
			
			# 记录当前时间
			A.last_trade_time = current_time
		else:
			print(f'一小时内有过交易，不再处理')
		
	# 死叉
	if macd[-2]>0 and macd[-1]<=0:
		print(f'进入死叉状态')
		# 如果 last_trade_time 为 None，或者距离上次成交时间超过1小时
		if A.last_trade_time is None or (current_time - A.last_trade_time).total_seconds() > A.trade_time:
			print(f'卖出{A.stock}股票，数量：{A.volume}，请保证有持仓!')
			passorder(24,1101,A.account,A.stock,14,-1,A.volume,2,C)
			
			# 记录当前时间
			A.last_trade_time = current_time
		else:
			print(f'一小时内有过交易，不再处理')
		